# Diffusion Models for Image Generation

## Introduction
* A diffusion model is an artificial intelligence/machine learning algorithm for image generation. 
* Diffusion models are behind several popular AI image generation tools, such as Dalle 3 and Stable Diffusion, which allow you to go from a text prompt to an image.
* **Overview**: Diffusion models are a type of deep neural network that learn how to add noise to a picture and then learn how to reverse that process to reconstruct a clear image.

## The Algorithm

### Part 1: Forward Diffusion (Classical)
* Given an image, add noise to the image by randomly selecting pixels to modify and then add a randomly selected noise to the pixels. Repeat this process over many time steps until the original image starts to disappear and there is just "tv static" remaining.
* The noise is selected randomly from a Gaussian distribution and is added via a Markov chain, meaning that the current state of the image only depends on its most recent state.
* Forward diffusion should end with an image which is just white noise and has no recognizable features.
* The amount of time needed for an image to be completely transformed to white noise is controlled by the **noise scheduler**
    * Controls the variance of the Gaussian distributions (higher variance means there is a higher chance of large noise being added each step)

### Part 2: Reverse Diffusion (AI)
* Start with the output of the forward diffusion process and then learn to remove the noise in structured and controlled manners in order to reconstruct the original image.
* This portion does use machine learning, specifically a convolutional neural network architecture called a U-Net
* Training Process:
    * Given the final forward diffusion image, predict the noise that was added during the last time step
    * Subtract the predicted noise from the image to get the state of the image in the second to last time step
    * Repeat for as many time steps as needed to reveal the features of the image
    * *The U-Net learns to minimize the MSE between the predicted noise and the actual noise*

## Conditional Diffusion
* Forward and reverse diffusion only train the model how to create images, conditional diffusion is how to take a text prompt an generate an image.
* Conditional diffusion starts with a text prompt which is embedded and then the embedding is paired with the image the prompt describes. The pair is then used to train the reverse diffusion process.
* **Self Attention Guidance:** How does a specific portion of the prompt change a specific portion of the image?
* **Classifier Free Guidance:** Amplifies the affect that certain words have on how the image is generated.

## Diffusion Denoising Probablistic Model (DDPM)
* The base diffusion model. It uses both a forward and reverse Markov chain, meaning that the model will need a large amount of time steps but tends to be reliable.
* Other models are avaliable that use different techniques to reduce the number of time steps.


## Pre-Trained Model

In [ ]:
###################
## INSTALLATIONS ##
###################
# Run the below line if needed to install diffusers. Change pip to pip3 if needed.
# Comment out after the first run.
#!pip3 install diffusers

#############
## IMPORTS ##
#############
# Import pre-built diffusion pipeline and schedulers
from diffusers import DDPMPipeline, DDIMScheduler, DDPMScheduler
import time
# Import torch and other utilities
import torch
import torchvision.transforms.functional as TF
from torchvision.utils import make_grid, save_image
#from PIL import Image

######################
## DEVICE SELECTION ##
######################
# Select device (GPU if available, else CPU)
device = "cuda" if torch.cuda.is_available() else "cpu"

####################################
## CHOOSE MODEL FROM HUGGING FACE ##
####################################
# In this code we will be looking at four different pre-trained diffusion models from Hugging Face. 
# These models are trained on different datasets and can generate different types of images. The first
# model is trained on MNIST digits, the second on CIFAR-10 images, the third on flower images, and the
# fourth on cat images. MNIST represents the smallest images (at 28x28 pixels) while the cat images
# are the largest (at 256x256 pixels). The models trained on CIFAR-10 and flowers generate images
# at 32x32 and 64x64 pixels respectively. Each model uses a UNet architecture but the complexity
# of the UNet (number of layers, channels, etc.) varies based on the dataset and image size.
model_ids = ["dvgodoy/ddpm-cifar10-32-mnist", "google/ddpm-cifar10-32" ,  "mrm8488/ddpm-ema-flower-64",  "google/ddpm-cat-256"]
names = ["MNIST Digits", "CIFAR-10 Images", "Flowers", "Cats"]

for i in range(len(model_ids)):
    # Start time measurement, get the current model id, and print the model name
    start = time.time()
    model_id = model_ids[i]
    print(names[i])

    ##############################################################
    ## LOAD THE DIFFUSION PIPELINE AND SCHEDULER FROM DIFFUSERS ##
    ##############################################################
    # The below code will load the DDPM pipeline with the pretrained weights and move it to the selected 
    # device (GPU or CPU). The pipeline includes the model and the scheduler (which defines the diffusion 
    # process). The model in this pipeline is a UNet-based model. We recently used U-net for image 
    # segmentation, but here it is used for denoising images at different time steps in the diffusion 
    # process.

    # For the scheduler, we will replace the default DDPM scheduler with a DDIM scheduler. DDIM is an 
    # alternative diffusion process which can generate decent quality images with fewer inference steps 
    # compared to DDPM. This makes the sampling process faster. You can also use the default DDPM scheduler if you want. Try both and see the difference in quality and speed. DDIM is not guaranteed to 
    # be better than DDPM, but it often is for many models.

    pipe = DDPMPipeline.from_pretrained(model_id).to(device)
    pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)  # or DDPMScheduler.from_config(...)

    ######################
    ## GENERATE IMAGES  ##
    ######################
    # Define number of images to generate and number of inference steps. Inference steps control the 
    # number of denoising steps in the diffusion process. More steps usually lead to better quality but 
    # take more time. Fewer steps are faster but may lead to lower quality. DDIM can often produce decent 
    # quality with fewer steps compared to DDPM. If you switch to DDPM scheduler, you may want to increase 
    # the number of steps for better quality.
    num_images = 16
    num_inference_steps = 50 

    # Generate images using the pipeline defined above. The autocast context is used to speed up inference 
    # on GPUs but this will work on CPU as well (without autocast).
    with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=torch.cuda.is_available()):
        out = pipe(batch_size=num_images, num_inference_steps=num_inference_steps)
    # Collect the generated images from the output of the pipeline
    images = out.images  # list of PIL images



    ##############################################
    ## SAVE GENERATED IMAGES AS A GRID PNG FILE ##
    ###############################################
    # Convert list[PIL.Image] -> torch.Tensor in [0,1], shape (N,C,H,W)
    batch = torch.stack([TF.to_tensor(img) for img in images])  # (N, 1 or 3, H, W)

    # Create a grid (e.g., 4x4 for 16 images) and save directly as PNG
    grid = make_grid(batch, nrow=4)  
    # Save the generated images as a grid image and save that as a PNG file. You can open
    # the PNG file to see the generated images. Note at visual studio will open the image in a new tab.
    save_image(grid, names[i]+".png")
    # End time measurement and print time taken
    end = time.time()
    print(f"Time taken: {end - start:.2f} seconds\n")



MNIST Digits


Loading pipeline components...:   0%|          | 0/2 [00:00<?, ?it/s]

An error occurred while trying to fetch /Users/butlerju/.cache/huggingface/hub/models--dvgodoy--ddpm-cifar10-32-mnist/snapshots/e2eb13c6ae24de41be4fbd24ef3b801f4a38a80e: Error no file named diffusion_pytorch_model.safetensors found in directory /Users/butlerju/.cache/huggingface/hub/models--dvgodoy--ddpm-cifar10-32-mnist/snapshots/e2eb13c6ae24de41be4fbd24ef3b801f4a38a80e.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.


  0%|          | 0/50 [00:00<?, ?it/s]

Time taken: 36.08 seconds

CIFAR-10 Images


...

Time taken: 98.55 seconds

Cats


Loading pipeline components...:   0%|          | 0/2 [00:00<?, ?it/s]

An error occurred while trying to fetch /Users/butlerju/.cache/huggingface/hub/models--google--ddpm-cat-256/snapshots/82ca0d5db4a5ec6ff0e9be8d86852490bc18a3d9: Error no file named diffusion_pytorch_model.safetensors found in directory /Users/butlerju/.cache/huggingface/hub/models--google--ddpm-cat-256/snapshots/82ca0d5db4a5ec6ff0e9be8d86852490bc18a3d9.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.


  0%|          | 0/50 [00:00<?, ?it/s]

Time taken: 1413.74 seconds



## Suggested References
* [Diffusion Models for AI Image Generation (Video)](https://www.youtube.com/watch?v=x2GRE-RzmD8)
    * Good, low math explanation
* [But how do AI images and videos actually work? | Guest video by Welch Labs (Video)](https://www.youtube.com/watch?v=iv-5mZ_9CPY&t=196s)
    * 3Blue1Brown but a guest speaker.
* [Diffusion Model from Scratch in Pytorch](https://towardsdatascience.com/diffusion-model-from-scratch-in-pytorch-ddpm-9d9760528946/)
    * Useful to work through if you want to get into how the different components work (U-Net, Noise Scheduler, etc.). However, the run times are significant without a GPU to get poor results.
* [Denoising Diffusion Probabilistic Models](https://arxiv.org/abs/2006.11239)
    * The original DDPM paper.
* [Understanding Deep Learning](https://udlbook.github.io/udlbook/)
    * See the chapter on diffusion models, though it is math heavy
* [Denoising Diffusion Implicit Models](https://arxiv.org/abs/2010.02502)
    * DIMM original paper
* [
Diffusion Models (DDPM & DDIM) - Easily explained!](https://www.youtube.com/watch?v=r4V0vLhYZIQ)
    * Walks through the differences between DDPM and DIMM but math heavy
